# Extra Experiments — candidate additions to the thesis

Self-contained. Recomputes everything from the cached PDC VAR(20) features
(`cache_pdc_var20/`) with the verified metrics layer. Use it to decide
which "extra results" are worth writing into the Word document.

**Which topics can run from the feature cache, and which cannot:**

| Topic | Runnable from cache? | Why |
|---|---|---|
| Variable seizure-prediction horizon (SPH) | yes (Exp A) | trim the preictal block tail |
| Expanding-window within-patient CV | yes (Exp B) | different chronological splits |
| Transfer-learning / patient calibration | yes (Exp C) | add the test patient's early data |
| Preictal sub-segmentation (early/mid/late) | yes (Exp D) | split each preictal block by position |
| Directed Transfer Function (DTF) | NO, needs re-extraction | requires raw VAR coefficients |
| Full 22+ channel montage | NO, needs re-extraction | more channels => recompute PDC |
| Source-space GC (beamforming/ICA) | NO, needs raw EEG | source reconstruction upstream |

Four of the seven run here in minutes; three require re-running feature
extraction from the raw EEG (hours) and are left as future work.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

import os, sys, warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# [path set by bootstrap] CODE   = Path(r"<repo>/Code")
# [path set by bootstrap] CODEV2 = Path(r"<repo>/this repository")
CACHE  = CODE / "cache_pdc_var20"
OUT    = CODEV2 / "outputs"; OUT.mkdir(exist_ok=True)
# sys.path.insert(0, str(CODEV2 / "src"))
from seizure_metrics import infer_seizure_groups
from seeds import set_global_seed
SEED=42; set_global_seed(SEED)
STEP_SEC=10; CAP_MULT,CAP_ABS=5,5000
EXCLUDED={"chb11","chb12","chb21"}

patients=sorted(p.name for p in CACHE.iterdir() if p.is_dir() and p.name.startswith("chb") and p.name not in EXCLUDED)
DATA={p:(np.load(CACHE/p/"features.npy").astype(np.float32), np.load(CACHE/p/"labels.npy").astype(int)) for p in patients}
print(len(patients),"patients loaded")

def cap_balance(X,y,seed):
    r=np.random.default_rng(seed); npre=int((y==1).sum()); ii=np.where(y==0)[0]
    if npre==0: return X[:0],y[:0]
    keep=min(len(ii),CAP_MULT*npre,CAP_ABS)
    sel=np.sort(np.concatenate([np.where(y==1)[0], r.choice(ii,keep,replace=False)])); return X[sel],y[sel]
def lr(): return Pipeline([("s",StandardScaler()),("c",LogisticRegression(max_iter=400,class_weight="balanced",C=0.1,random_state=SEED))])
def skill(ap,prev): return (ap-prev)/(1-prev) if prev<1 else np.nan


21 patients loaded


## Exp A - Variable seizure-prediction horizon (SPH)

The cached preictal blocks already exclude the fixed 5-minute SPH. A longer SPH
is simulated by trimming the block tail (windows closest to onset). Answers the
horizon-sensitivity limitation at near-zero cost.


In [2]:
def trim_tail_minutes(y, drop_min):
    drop=int(drop_min*6); y2=y.copy(); i=0; n=len(y)
    while i<n:
        if y[i]==1:
            j=i
            while j<n and y[j]==1: j+=1
            for d in range(max(i,j-drop),j): y2[d]=0
            i=j
        else: i+=1
    return y2

def lopo_ap(relabel=None):
    aps=[]; prevs=[]; aucs=[]
    for i,test in enumerate(patients):
        Xtr,ytr=[],[]
        for p in patients:
            if p==test: continue
            X,y=DATA[p]; y=relabel(y) if relabel else y
            if y.sum()==0: continue
            Xc,yc=cap_balance(X,y,SEED+i)
            if len(yc): Xtr.append(Xc); ytr.append(yc)
        Xtr=np.vstack(Xtr); ytr=np.concatenate(ytr)
        m=lr().fit(Xtr,ytr)
        X,y=DATA[test]; y=relabel(y) if relabel else y
        if len(np.unique(y))<2: continue
        prob=m.predict_proba(X)[:,1]
        aucs.append(roc_auc_score(y,prob)); aps.append(average_precision_score(y,prob)); prevs.append(float((y==1).mean()))
    return np.mean(aucs),np.mean(aps),np.mean(prevs)

rows=[]
for sph,extra in [(5,0),(10,5),(15,10)]:
    a,ap,pv=lopo_ap(None if extra==0 else (lambda y,e=extra: trim_tail_minutes(y,e)))
    rows.append({"SPH_min":sph,"AUC":round(a,3),"AUC_PR":round(ap,3),"prevalence":round(pv,3),"Skill":round(skill(ap,pv),3)})
expA=pd.DataFrame(rows); expA.to_csv(OUT/"expA_variable_sph.csv",index=False)
print(expA.to_string(index=False))


 SPH_min   AUC  AUC_PR  prevalence  Skill
       5 0.550   0.382       0.337  0.069
      10 0.541   0.302       0.265  0.051
      15 0.534   0.226       0.195  0.039


## Exp B - Expanding-window within-patient cross-validation

Defends the single 80/20 within-patient split: each patient is evaluated over
3 expanding folds (train first 40/60/80 %, test next block); mean compared to
the single split.


In [3]:
def within_expanding(folds=(0.4,0.6,0.8)):
    rows=[]
    for p in patients:
        X,y=DATA[p]; n=len(y); aps=[]
        for f in folds:
            k=int(n*f); ke=int(n*min(f+0.2,1.0))
            Xtr,ytr,Xte,yte=X[:k],y[:k],X[k:ke],y[k:ke]
            if ytr.sum()==0 or yte.sum()==0 or len(np.unique(yte))<2: continue
            m=lr().fit(Xtr,ytr); aps.append(average_precision_score(yte,m.predict_proba(Xte)[:,1]))
        if aps: rows.append({"patient":p,"aucpr_expanding":np.mean(aps),"n_folds":len(aps)})
    return pd.DataFrame(rows)

def within_single():
    rows=[]
    for p in patients:
        X,y=DATA[p]; n=len(y); k=int(n*0.8)
        if y[:k].sum()==0 or y[k:].sum()==0 or len(np.unique(y[k:]))<2: continue
        m=lr().fit(X[:k],y[:k]); rows.append({"patient":p,"aucpr_single":average_precision_score(y[k:],m.predict_proba(X[k:])[:,1])})
    return pd.DataFrame(rows)

exp=within_expanding().merge(within_single(),on="patient"); exp.to_csv(OUT/"expB_expanding_window.csv",index=False)
print("Within-patient AUC-PR - single 80/20: %.3f | expanding-window mean: %.3f (n=%d)"
      %(exp.aucpr_single.mean(), exp.aucpr_expanding.mean(), len(exp)))


Within-patient AUC-PR - single 80/20: 0.514 | expanding-window mean: 0.558 (n=17)


## Exp C - Transfer learning / patient calibration (the deployment story)

Three settings, all evaluated on each patient's last 40 % of windows:
cross-patient only; calibrated (cross-patient pool + this patient's first 60 %);
within-patient only (first 60 %).


In [4]:
def transfer_experiment():
    rows=[]
    for i,test in enumerate(patients):
        X,y=DATA[test]; n=len(y); split=int(n*0.6)
        Xte,yte=X[split:],y[split:]
        if len(np.unique(yte))<2 or y[:split].sum()==0: continue
        Xp,yp=[],[]
        for p in patients:
            if p==test: continue
            Xc,yc=cap_balance(*DATA[p],SEED+i)
            if len(yc): Xp.append(Xc); yp.append(yc)
        Xp=np.vstack(Xp); yp=np.concatenate(yp)
        m_cross=lr().fit(Xp,yp)
        m_cal=lr().fit(np.vstack([Xp,X[:split]]), np.concatenate([yp,y[:split]]))
        m_within=lr().fit(X[:split],y[:split])
        ap=lambda mm: average_precision_score(yte,mm.predict_proba(Xte)[:,1])
        rows.append({"patient":test,"cross":ap(m_cross),"calibrated":ap(m_cal),"within":ap(m_within)})
    return pd.DataFrame(rows)

expC=transfer_experiment(); expC.to_csv(OUT/"expC_transfer_calibration.csv",index=False)
print("Mean AUC-PR on held-out last 40%% of each patient (n=%d):"%len(expC))
print("  Cross-patient only : %.3f"%expC.cross.mean())
print("  Calibrated (hybrid): %.3f"%expC.calibrated.mean())
print("  Within-patient only: %.3f"%expC.within.mean())


Mean AUC-PR on held-out last 40% of each patient (n=20):
  Cross-patient only : 0.431
  Calibrated (hybrid): 0.438
  Within-patient only: 0.449


## Exp D - Preictal sub-segmentation (is the preictal window homogeneous?)

Motivated by Batista et al. (2024). Each preictal block is split into early,
middle and late thirds (by distance to onset); LOPO AUC-PR is measured using
only that third as the positive class.


In [5]:
def third_mask(y, which):
    y2=np.zeros_like(y); i=0; n=len(y)
    while i<n:
        if y[i]==1:
            j=i
            while j<n and y[j]==1: j+=1
            L=j-i; a=i+{0:0,1:L//3,2:2*L//3}[which]; b=i+{0:L//3,1:2*L//3,2:L}[which]
            for d in range(a,b): y2[d]=1
            i=j
        else: i+=1
    return y2

rows=[]
for name,w in [("early (far from onset)",0),("middle",1),("late (near onset)",2)]:
    aps=[]
    for test in patients:
        X,y=DATA[test]; yt=third_mask(y,w); mask=~((y==1)&(yt==0)); yy=yt[mask]
        if yy.sum()==0 or len(np.unique(yy))<2: continue
        Xtr,ytr=[],[]
        for p in patients:
            if p==test: continue
            Xp,yp=DATA[p]; ytp=third_mask(yp,w); mp=~((yp==1)&(ytp==0))
            if ytp[mp].sum()==0: continue
            Xc,yc=cap_balance(Xp[mp],ytp[mp],SEED)
            if len(yc): Xtr.append(Xc); ytr.append(yc)
        Xtr=np.vstack(Xtr); ytr=np.concatenate(ytr)
        m=lr().fit(Xtr,ytr); aps.append(average_precision_score(yy, m.predict_proba(X[mask])[:,1]))
    rows.append({"preictal_third":name,"mean_AUC_PR":round(float(np.mean(aps)),3),"n":len(aps)})
expD=pd.DataFrame(rows); expD.to_csv(OUT/"expD_preictal_thirds.csv",index=False)
print(expD.to_string(index=False))


        preictal_third  mean_AUC_PR  n
early (far from onset)        0.189 21
                middle        0.185 21
     late (near onset)        0.193 21


## Summary - which are worth adding to the Word document?

- Report it if the result is stable and either supports a claim already made
  (Exp B validating the single split) or adds a defensible new finding
  (Exp C: calibration recovers performance).
- Leave as future work if the effect is null/noisy or needs the re-extraction
  experiments (DTF, full montage, source space).

Strongest candidates are usually Exp C (deployment story: cross < calibrated ~=
within) and Exp B (defends the single-split number). Exp A and D are worth one
sentence each only if the trend is monotone.
